# Emission mapping procedure to disaggregate NACE Rev2 emissions
@author Florian Wirth

When computing the impact of carbon prices, we need sectoral emissions according to the NACE classification to work with the IO framework. However, we also need these emissions disaggregated into their different IPCC2006 / common reporting framework (CRF) components, because the regulation applies not at the level of sectors but at that of emission types according to the CRF classification. 

This gives rise to a biproportional balancing problem: We can use the NACE and CRF emissions as row and column totals and balance an initial concordance matrix with a RAS algorithm. 


## Load MRIO data

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
import yaml
import logging
import eurostat
from mrio_toolbox import MRIO, Part, extract_MRIO
from mrio_toolbox.utils.savers._to_nc import save_to_nc 
# Suppress warnings about duplicate dimension names specific UserWarnings from xarray.namedarray
warnings.filterwarnings("ignore", category=UserWarning, module=r"xarray\.namedarray\.core")

logging.basicConfig(level=logging.INFO)

year = 2024
edition = 26
version = "eu27" 

filepath = os.path.abspath(f"../data/figaro io/formatted io/figaro_year{year}.nc")
grouping_file_path = os.path.abspath(f"../data/figaro io/figaro{edition}ed_grouping_{version}.yaml")
output_filepath = os.path.abspath(f"../data/figaro io/formatted io/figaro_year{year}_aggregated_{version}.nc")

if not os.path.isfile(filepath):
    figaro = extract_MRIO(source= "../data/figaro io/raw io/",
                          table = "figaro",
                          year = year,
                          extraction_kwargs= {"edition": edition, "sut" :"supply"})
    save_to_nc(figaro, filepath)
else:
    figaro = MRIO(file = filepath)

    
groupings = yaml.safe_load(open(grouping_file_path))
figaro.set_groupings(groupings)
figaro.aggregate(on = "countries")

INFO:mrio_toolbox.utils.loaders._nc_loader:Load MRIO data from /home/florian/Wissenschaft/eu-ets-inflation-impact/data/figaro io/formatted io/figaro_year2024.nc
INFO:mrio_toolbox._parts._Part:Checking if a reformatting is needed.
INFO:mrio_toolbox._parts._Part:Reformat the Part
INFO:mrio_toolbox.mrio:Add part t to MRIO table
INFO:mrio_toolbox.mrio:Add part t to MRIO table
INFO:mrio_toolbox._parts._Part:Checking if a reformatting is needed.
INFO:mrio_toolbox._parts._Part:Reformat the Part
INFO:mrio_toolbox.mrio:Add part y to MRIO table
INFO:mrio_toolbox.mrio:Add part y to MRIO table
INFO:mrio_toolbox._parts._Part:Checking if a reformatting is needed.
INFO:mrio_toolbox._parts._Part:Reformat the Part
INFO:mrio_toolbox.mrio:Add part va to MRIO table
INFO:mrio_toolbox.mrio:Add part va to MRIO table
INFO:mrio_toolbox._parts._Part:Checking if a reformatting is needed.
INFO:mrio_toolbox._parts._Part:Reformat the Part
INFO:mrio_toolbox.mrio:Add part vay to MRIO table
INFO:mrio_toolbox.mrio:Add 

## Load emission data and concordance table

In [2]:
# get household transport emissions from eurostat
q_eurostat = eurostat.get_data_df("env_ac_ainah_r2", filter_pars={'airpol': ['CO2', 'N2O', 'CH4'], 'unit': 'THS_T'})
q_eurostat = q_eurostat.rename(columns={"geo\\TIME_PERIOD": "country"})
q_eurostat = q_eurostat[~(q_eurostat["country"].isin(["EU27_2020", "RS", "TR"]))]
#q_eurostat = q_eurostat[~(q_eurostat["nace_r2"].isin(["H50", "H51"]))] # exclude aviation and water transport
q_eurostat = q_eurostat[q_eurostat["airpol"] == "CO2"]
nace = q_eurostat["nace_r2"].unique()
q_eurostat = q_eurostat.replace({"EL": "GR"})
q_eurostat = q_eurostat[["nace_r2", "country", str(year)]]

In [3]:
q_crf = pd.read_csv("../data/emission data/crf_2026/CRT_emissions_processed.csv")
q_crf = q_crf.rename(columns={"area (ISO2)": "country", "entity": "airpol"})
q_crf = q_crf[q_crf["airpol"] == "CO2"]
q_crf = q_crf[["category", "country", str(year)]]
q_crf = q_crf[~(q_crf["country"].isin(["GB", "LI"]))]
q_crf = q_crf[~(q_crf["category"].isin(["M.0.EL", "M.AG"]))]
#q_crf = q_crf[~(q_crf["category"].isin(["1.D.1.a", "1.D.1.b"]))] # Exclude International aviation and water transport
q_crf = q_crf.dropna()
q_crf = q_crf.set_index(["category", "country"])

In [4]:
# load concordance table
concordance = pd.read_excel("../data/emission data/mapping documentation/mapping_crf_Figaro.xlsx")
concordance["Category Code (IPCC2006)"] = concordance["Category Code (IPCC2006)"].astype("string")
concordance = concordance.set_index("Category Code (IPCC2006)")
concordance = concordance.drop(columns=["Category Name"])
concordance = concordance.dropna(subset=["Economic sector"])

sector_names = pd.read_excel("../data/emission data/mapping documentation/mapping_crf_Figaro.xlsx", sheet_name="Sectors_Figaro")
sector_names = sector_names.set_index("eurostat code")
mapping = sector_names["sector name"].to_dict()
q_eurostat["sector"] = q_eurostat["nace_r2"].map(mapping)
q_eurostat = q_eurostat.dropna()

countries = list(q_eurostat["country"].unique())
sectors = list(q_eurostat["sector"].unique())
crf_labs = list(concordance.index)

q_eurostat = q_eurostat.set_index(["sector", "country"])
q_eurostat = q_eurostat.drop(columns=["nace_r2"])

## Construct country totals

In [5]:
# construct row and column totals
crf_totals = Part(labels = {"0": {"countries" : countries, "crf_labs" : crf_labs}}, name = "crf_totals")
nace_totals= Part(labels = {"0": {"countries" : countries, "sectors" : sectors}}, name = "nace_totals")

In [6]:
for country in countries: 
    for crf_lab in crf_labs:
        try: 
            value = q_crf.loc[(crf_lab, country)].item()
        except KeyError:
            value = 0 # This country doesn't have emissions in that category
        crf_totals[country, crf_lab] = value
    for sector in sectors:
        nace_totals[country, sector] = q_eurostat.loc[(sector,country)].item() 

## Translate system boundaries with bridging items
We see that we have to make some adjustments in order to translate the system boundaries from the territorial emissions to the residential emissions. Wecan use the bridging items published by eurostat. 

In [7]:
bridge = eurostat.get_data_df("env_ac_aibrid_r2", filter_pars={'airpol': ['CO2'], 'unit': 'THS_T'})
labels = eurostat.get_dic("env_ac_aibrid_r2", "INDIC_ENV", full=False, frmt="dict")
bridge["bridge_item"] = bridge["indic_env"].map(labels)
bridge = bridge.rename(columns={"geo\\TIME_PERIOD": "country"})
bridge = bridge[~(bridge["country"].isin(["EU27_2020", "RS", "TR", "CH"]))]
bridge = bridge[bridge["airpol"] == "CO2"]
bridge = bridge.replace({"EL": "GR"})
bridge = bridge[["indic_env", "country", str(year)]]
bridge = bridge.set_index(["country", "indic_env"])

In [8]:
labels

{'AEMIS_RES': 'Air emissions by resident units (production activities and households)',
 'AEMIS_RES_ABR': 'Air emissions by resident units released from use of fuel purchased abroad - total',
 'AEMIS_RES_ABR_FWTR': 'Air emissions by resident units released from use of fuel purchased abroad  - fishing vessels',
 'AEMIS_RES_ABR_LTR': 'Air emissions by resident units released from use of fuel purchased abroad  - land transport',
 'AEMIS_RES_ABR_WTR': 'Air emissions by resident units released from use of fuel purchased abroad  - water transport',
 'AEMIS_RES_ABR_ATR': 'Air emissions by resident units released from use of fuel purchased abroad  - air transport',
 'AEMIS_TER_NRES': 'Air emissions by non-resident units released from use of fuel purchased on the territory (only if included in national totals according to international conventions) - total',
 'AEMIS_TER_NRES_LTR': 'Air emissions by non-resident units released from use of fuel purchased on the territory (only if included in nati

In [9]:
for country in countries: 
    try:
        crf_totals[country, "1.A.3.d"] += bridge.loc[country, "AEMIS_RES_ABR_FWTR"].item() # add fishing vessels to 'domestic navigation'
        
        land_travel_change = bridge.loc[country, "AEMIS_RES_ABR_LTR"].item() - bridge.loc[country, "AEMIS_TER_NRES_LTR"].item()
        air_travel_change = bridge.loc[country, "AEMIS_RES_ABR_ATR"].item() - bridge.loc[country, "AEMIS_TER_NRES_ATR"].item()
        water_travel_change = bridge.loc[country, "AEMIS_RES_ABR_WTR"].item() - bridge.loc[country, "AEMIS_TER_NRES_WTR"].item()
        
        # add land travel changes proportionally to all road transport emissions
        road_transport_emissions = crf_totals[country, "1.A.3.b.i"] + crf_totals[country, "1.A.3.b.ii"] + crf_totals[country, "1.A.3.b.iii"] + crf_totals[country, "1.A.3.b.iv"] 
        car_weight = crf_totals[country, "1.A.3.b.i"] / road_transport_emissions
        ldtruck_weight = crf_totals[country, "1.A.3.b.ii"] / road_transport_emissions
        hdtruck_weight = crf_totals[country, "1.A.3.b.iii"] / road_transport_emissions
        motorcycle_weight = crf_totals[country, "1.A.3.b.iv"] / road_transport_emissions
        
        crf_totals[country, "1.A.3.b.i"] += land_travel_change * car_weight
        crf_totals[country, "1.A.3.b.ii"] += land_travel_change * ldtruck_weight
        crf_totals[country, "1.A.3.b.iii"] += land_travel_change * hdtruck_weight
        crf_totals[country, "1.A.3.b.iv"] += land_travel_change * motorcycle_weight 
        
        crf_totals[country, "1.D.1.a"] += air_travel_change # add air travel change to international aviation
        crf_totals[country, "1.D.1.b"] += water_travel_change # add water travel change to international navigation
        
        
        if crf_totals[country, "1.D.1.a"] < 0: 
            print(f"warning, international air travel change in country {country} is {crf_totals[country, "1.D.1.a"]}")
        if crf_totals[country, "1.D.1.b"] < 0: 
            print(f"warning, international water travel change in country {country} is {crf_totals[country, "1.D.1.b"]}")
        
    except KeyError as e:
        print(country, "doesn't have emissions for", e, ", skipping")

RO doesn't have emissions for 'AEMIS_TER_NRES_LTR' , skipping
SK doesn't have emissions for 'AEMIS_RES_ABR_LTR' , skipping


In [10]:
crf_totals["MT"].to_pandas().sort_values(by = 0, ascending = False)

0
countries crf_labs                
MT        1.D.1.b      6788.804570
          1.D.1.a      5220.112028
          1.A.1.a       736.597241
          1.A.3.b.i     430.865679
          1.A.3.b.iii   130.538118
...                            ...
          5.B.2           0.000000
          5.C.2           0.000000
          5.D.1           0.000000
          5.D.2           0.000000
          5.D.3           0.000000

[155 rows x 1 columns]

## Consistency check: mono-sector CRF categories vs. NACE supply

Some CRF categories map to exactly one NACE sector in the concordance table (e.g. `1.A.1.b` -> `Manufacture of coke and refined petroleum products`). For these, the full CRF row total has to be absorbed by that single NACE column alone, so if the CRF total for a mono-sector category (summed across all mono-sector categories sharing the same NACE sector) exceeds the NACE column's total supply, RAS can't converge 


In [11]:
# Identify CRF categories that map to exactly one NACE sector in the concordance table
sector_lists = concordance.apply(lambda row: row.dropna().tolist(), axis=1)
n_sectors = sector_lists.apply(len)
mono_sectors = sector_lists[n_sectors == 1].apply(lambda x: x[0])
mono_sectors.name = "sector"
mono_sectors.index.name = "crf_lab"

print(f"{len(mono_sectors)} / {len(sector_lists)} CRF categories map to exactly one NACE sector:")
mono_sectors.to_frame()

52 / 155 CRF categories map to exactly one NACE sector:


,sector
crf_lab,
1.A.1.b,Manufacture of coke and refined petroleum prod...
1.A.1.c.ii,Mining and quarrying
1.A.1.c.iii,"Electricity, gas, steam and air conditioning s..."
1.A.2.e,Manufacture of food products; beverages and to...
1.A.2.f,Manufacture of other non-metallic mineral prod...
1.A.3.b.iv,Transport activities by households
1.A.3.c,Land transport and transport via pipelines
1.A.3.e.i,Land transport and transport via pipelines
1.A.4.c.iii,Fishing and aquaculture


In [12]:
# Pull the row (CRF) and column (NACE) totals into flat, long-format DataFrames
crf_df = crf_totals.to_pandas().reset_index()
crf_df.columns = ["countries", "crf_labs", "value"]

nace_df = nace_totals.to_pandas().reset_index()
nace_df.columns = ["countries", "sectors", "value"]

# For every country, sum the CRF totals of all mono-sector categories that share the same NACE sector  (several mono-sector 
# categories can point at the same sector, e.g. all the 2.B.x categories mapping to "Manufacture of chemicals and chemical products" 
# -- they all compete for the same column, so their demand adds up)
crf_mono = crf_df[crf_df["crf_labs"].isin(mono_sectors.index)].copy()
crf_mono["sector"] = crf_mono["crf_labs"].map(mono_sectors)

demand = (
    crf_mono
    .groupby(["countries", "sector"])["value"]
    .sum()
    .reset_index()
    .rename(columns={"value": "crf_demand"})
)

# Compare summed CRF demand against the NACE column's total supply
supply = nace_df.rename(columns={"sectors": "sector", "value": "nace_supply"})
consistency_check = demand.merge(supply, on=["countries", "sector"], how="left")
consistency_check["ratio"] = consistency_check["crf_demand"] / consistency_check["nace_supply"]
consistency_check = consistency_check.sort_values("ratio", ascending=False).reset_index(drop=True)
infeasible_pairs = consistency_check[consistency_check["ratio"] > 1]

print(f"{len(infeasible_pairs)} country/sector pairs are structurally infeasible "
      f"(CRF demand exceeds NACE supply):")
infeasible_pairs = infeasible_pairs.set_index(["countries", "sector"])
infeasible_pairs


29 country/sector pairs are structurally infeasible (CRF demand exceeds NACE supply):


crf_demand  \
countries sector                                                             
NO        Manufacture of chemicals and chemical products        908.592073   
CZ        Manufacture of coke and refined petroleum products    461.853936   
IS        Manufacture of food products; beverages and tob...     62.196706   
DK        Fishing and aquaculture                               848.007799   
SK        Manufacture of other non-metallic mineral products   3026.348034   
          Manufacture of coke and refined petroleum products   1753.801793   
FI        Public administration and defence; compulsory s...    617.113576   
RO        Public administration and defence; compulsory s...   1147.266588   
SI        Mining and quarrying                                  127.843803   
NL        Manufacture of other non-metallic mineral products   1764.550440   
PT        Manufacture of other non-metallic mineral products   5365.228652   
CZ        Manufacture of other non-metallic mineral products   4531.296296   
HU        Fishing and aquaculture                                 6.336183   
FR        Fishing and aquaculture                              1209.823646   
SK        Manufacture of food products; beverages and tob...    321.178056   
CZ        Public administration and defence; compulsory s...    242.253003   
NO        Manufacture of basic metals                          4628.620875   
HR        Manufacture of food products; beverages and tob...    288.037136   
FI        Manufacture of other non-metallic mineral products    967.134706   
PL        Manufacture of other non-metallic mineral products  18024.777170   
IT        Manufacture of coke and refined petroleum products  17182.244307   
IS        Manufacture of basic metals                          1824.419256   
NL        Fishing and aquaculture                               326.044701   
HR        Manufacture of other non-metallic mineral products   2341.940962   
EE        Manufacture of basic metals                             3.070521   
AT        Manufacture of coke and refined petroleum products   2620.210000   
HR        Manufacture of coke and refined petroleum products    702.891708   
DK        Manufacture of coke and refined petroleum products    848.682237   
GR        Manufacture of other non-metallic mineral products   5461.244916   

                                                              nace_supply  \
countries sector                                                            
NO        Manufacture of chemicals and chemical products          0.00000   
CZ        Manufacture of coke and refined petroleum products     23.39015   
IS        Manufacture of food products; beverages and tob...     15.94773   
DK        Fishing and aquaculture                               382.60555   
SK        Manufacture of other non-metallic mineral products   1414.47783   
          Manufacture of coke and refined petroleum products    905.12821   
FI        Public administration and defence; compulsory s...    335.98703   
RO        Public administration and defence; compulsory s...    626.35657   
SI        Mining and quarrying                                   76.85285   
NL        Manufacture of other non-metallic mineral products   1113.63194   
PT        Manufacture of other non-metallic mineral products   4019.61631   
CZ        Manufacture of other non-metallic mineral products   3437.92655   
HU        Fishing and aquaculture                                 4.92285   
FR        Fishing and aquaculture                               944.76445   
SK        Manufacture of food products; beverages and tob...    254.55395   
CZ        Public administration and defence; compulsory s...    192.33606   
NO        Manufacture of basic metals                          3824.30498   
HR        Manufacture of food products; beverages and tob...    243.09509   
FI        Manufacture of other non-metallic mineral products    823.64007   
PL        Manufacture of oth

Interestingly, while many of these are minor deviations, it is specifically Norways chemical industry which just got left out of eurostat emissions completely. 

In [13]:
# Increase the NACE sectors emission to match CRF demand

for country, sector in infeasible_pairs.index:
    value = infeasible_pairs.loc[country, sector]["crf_demand"]
    nace_totals[country, sector] = value

### Manual adjustments
There are some cases of inconsistency that can not be found with this method, because multiple sectors are involved, but will later turn up as a failure of the balancing procedure. I will manually adjust them here.

#### 1. Luxembourgs chemical industry
Luxembourg has 71 kt CO2 emissions from 1.A.2.c which can map to three sectors (chemics, pharmaceutics and petroleum). Since the chemical industry is way bigger than the pharmaceutical one both in terms of (underreported nace) emissions and economic size, I will attribute the 1.A.2.c emissions to the chemics nace sector.

In [14]:
x = figaro.t.sum(axis = 1) + figaro.y.sum(axis = 1)
print("economic size:")
display(x["LU",["Manufacture of chemicals and chemical products",
                   "Manufacture of basic pharmaceutical products and pharmaceutical preparations", 
                   "Manufacture of coke and refined petroleum products"]].to_pandas())
print()
print("emissions:")
display(crf_totals["LU", "1.A.2.c"].to_pandas())
nace_totals["LU", ["Manufacture of chemicals and chemical products",
                   "Manufacture of basic pharmaceutical products and pharmaceutical preparations", 
                   "Manufacture of coke and refined petroleum products"]].to_pandas()

economic size:


0
countries sectors                                                    
LU        Manufacture of chemicals and chemical products      558.241
          Manufacture of basic pharmaceutical products an...   13.080
          Manufacture of coke and refined petroleum products    0.000


emissions:


,,0
countries,crf_labs,
LU,1.A.2.c,89.258531


0
countries sectors                                                    
LU        Manufacture of chemicals and chemical products      5.56308
          Manufacture of basic pharmaceutical products an...  0.01880
          Manufacture of coke and refined petroleum products  0.00000

In [15]:
nace_totals["LU", "Manufacture of chemicals and chemical products"] = crf_totals["LU", "1.A.2.c"]

## Remove residual differences

In order to be able to do the RAS, we will distribute the residual proportionally to all the sectors. For some countries, like Malta, this will be quite a big operation. However, we are still true to the eurostat air emission accounts per sector which have the most institutional information and should be most reliable for IO analysis. The distribution of emission categories should not be changed by the scaling operation. 

In [16]:
crf_t = [crf_totals[country].sum() for country in countries]
sector_t = [nace_totals[country].sum() for country in countries]
difference = pd.DataFrame({
    "countries":countries,
    "nace_totals":sector_t,
    "crf_totals":crf_t, 
    })
difference["diff"] =  difference["nace_totals"]- difference["crf_totals"]
difference["rel_diff"] = ( difference["nace_totals"] -difference["crf_totals"]) / difference["nace_totals"]
difference["abs_rel_diff"] = abs(difference["rel_diff"])
difference = difference.set_index("countries")
difference.sort_values(by = "abs_rel_diff", ascending = False)

,nace_totals,crf_totals,diff,rel_diff,abs_rel_diff
countries,,,,,
MT,6421.072500,13719.237780,-7298.165280,-1.136596,1.136596
NL,141140.281561,180177.389169,-39037.107609,-0.276584,0.276584
GR,58735.341016,74721.824807,-15986.483791,-0.272178,0.272178
BE,88803.234370,111678.947395,-22875.713025,-0.257600,0.257600
LU,8622.323951,10723.942315,-2101.618364,-0.243742,0.243742
IS,5282.364382,6521.380574,-1239.016192,-0.234557,0.234557
RO,88978.375178,68144.122728,20834.252451,0.234150,0.234150
CY,7266.783770,8904.217907,-1637.434137,-0.225331,0.225331
ES,225332.586070,272287.060760,-46954.474690,-0.208379,0.208379


In [17]:
crf_weights = crf_totals.alias()
for country in countries: 
    crf_weights[country] =  crf_totals[country] / crf_totals.sum(on = "crf_labs")[country]
    crf_totals[country] += crf_weights[country] * difference.loc[country, "diff"]

## RAS algorithm

In [18]:
# Construct matrix and fill with ones for concordance entries
Q = Part(labels = {"0": {"countries" : countries, "crf_labs" : crf_labs}, "1": {"countries" : countries, "sectors" : sectors}}, name = "Q")

#Q["all", "all"] = 1
for country in countries:
    for crf_lab in crf_labs:
        secs = concordance.loc[crf_lab].dropna()
        Q[[country, crf_lab],[country,secs]] = 1

In [19]:
def ras(matrix_part, row_tot_part, col_tot_part, n_iter = 100, rel_tol = 0.1):
    """
    X: Part object
        The outcome matrix that needs to be re-balanced
    row_tot: Part object
        The (vertical) row totals used for balancing
    col_tot: Part object
        The (horizontal) column totals used for balancing
    n_iter: int
        How many iterations should be balanced, before aborting
    rel_tol: 
        What should be the relative tolerance of matching between the given
        and balanced totals before returning the matrix
    """
    
    
    X = matrix_part.data
    row_tot = row_tot_part.data
    col_tot = col_tot_part.data
    
        # Apply floor to row and column target vectors
    epsilon = 1e-6
    row_tot = np.where(row_tot == 0, epsilon, row_tot)
    col_tot = np.where(col_tot == 0, epsilon, col_tot)
    
    for i in range (n_iter):

        row_sum =  X.sum(axis=1)
        row_scaling_factor = np.divide(row_tot, row_sum, out=np.ones_like(row_tot), where=row_sum != 0)
        X = X * row_scaling_factor[:, None]
        if i > 1: max_row_scaling_factor_old = max_row_scaling_factor
        max_row_scaling_factor = max(row_scaling_factor)
        
        col_sum = X.sum(axis = 0)
        column_scaling_factor = np.divide(col_tot, col_sum, out=np.ones_like(col_tot), where=col_sum != 0)
        X = X * column_scaling_factor[None,:]
        
        print(f"Iteration {i}: max. row_scaling_factor = {max_row_scaling_factor}")

        # Test if row_sum stays the same over multiple iterations -> We either converged or we are stuck
        if i > 1 and np.isclose(max_row_scaling_factor, max_row_scaling_factor_old, rtol=0.00001): 
            print("Row scaling factor doesn't change anymore. We either converged or we are stuck")
            matrix_part.data = X
            row_scaling_factor_part = row_tot_part.alias()
            col_scaling_factor_part = col_tot_part.alias()
            
            row_scaling_factor_part.data = row_scaling_factor
            col_scaling_factor_part.data = column_scaling_factor
            
            return matrix_part, row_scaling_factor_part, col_scaling_factor_part
        
        
        
    matrix_part.data = X
    return matrix_part, row_tot, col_tot

In [20]:
#output = ras (Q["DE", "DE"], crf_totals["DE"], nace_totals["DE"], 100, 0.01)
output = ras (Q, crf_totals, nace_totals, 100, 0.01)

Iteration 0: max. row_scaling_factor = 71777.37984738998
Iteration 1: max. row_scaling_factor = 34198662.75208789
Iteration 2: max. row_scaling_factor = 17.19922991852325
Iteration 3: max. row_scaling_factor = 17.070203849032396
Iteration 4: max. row_scaling_factor = 17.06620779486039
Iteration 5: max. row_scaling_factor = 17.06608416860112
Row scaling factor doesn't change anymore. We either converged or we are stuck


In [21]:
# After the operation, the NACE dimension is always balanced, because that's the last thing that we do. 
output[0].sum(axis = 0).to_pandas()

0
countries sectors                                                       
AT        Crop and animal production, hunting and related...  704.545360
          Forestry and logging                                 99.526840
          Fishing and aquaculture                               2.109330
          Mining and quarrying                                497.330420
          Manufacture of food products; beverages and tob...  865.819110
...                                                                  ...
SK        Activities of membership organisations               92.800400
          Repair of computers and personal and household ...    4.829430
          Other personal service activities                    11.944490
          Activities of households as employers; undiffer...    0.000001
          Activities of extraterritorial organisations an...    0.000000

[1943 rows x 1 columns]

But the IPCC categories don't line up....
The row scaling factor will tell us which IPCC category departs most from the data.

In [22]:
nace_totals.to_pandas().to_csv("nace_totals.csv")

In [23]:
crf_totals.to_pandas().to_csv("crf_totals.csv")

In [24]:
too_constrained_ipcc_categories = output[1].to_pandas()
too_constrained_ipcc_categories = too_constrained_ipcc_categories[0].rename("row_scaling_factor")
too_constrained_ipcc_categories = too_constrained_ipcc_categories.sort_values( ascending = False)
too_constrained_ipcc_categories.to_csv("too_constrained_ipcc_categories.csv")
too_constrained_ipcc_categories.head(50)


countries  crf_labs    
MT         1.D.1.b         17.066084
           1.A.3.d         17.066084
BE         1.A.3.d          5.337774
           1.D.1.b          5.337774
CY         1.A.3.a          5.044414
           1.D.1.a          5.044414
           1.D.1.b          5.019019
           1.A.3.d          5.019019
DE         2.E.1            4.367838
           2.E.2            4.367838
           2.E.3            4.367838
           2.E.5            4.367838
           2.E.4            4.367838
           2.D.4            4.362878
           1.A.2.g.viii     4.358645
ES         1.D.1.b          3.985121
           1.A.3.d          3.985121
DE         1.A.2.g          3.931520
           1.A.2.g.vii      3.930253
           1.B.1.a.i        3.872629
           1.B.1.a.i.2      3.872629
           1.B.1.a.i.1      3.872629
           1.B.2.b.i        3.872629
           1.B.2.b.iii      3.872629
           1.B.1.a          3.872629
           1.B.2.a.i        3.872629
           1.B

For Malta, the problem is with the bridge items from eurostat, there is still a considerable residual there, which Eurostat also acknowledges. Luxembourg had a problem with the chemical industry which I resolved above. I can not manually adjust all the imbalances, there are real problems with the data. I will therefore stick with the Eurostat balance for now, especially since I need to work with emission intensities of sectors and I assume that the eurostat data is better balanced in terms of emission intensity than if I would have the CRF data have "the last word" in the distribution of sectoral emissions. 

## Move emissions to intermediate and final demand

In [25]:
hh_sectors = ["Heating/cooling activities by households","Transport activities by households","Other activities by households"]
sectors = [s for s in sectors if s not in hh_sectors]

In [26]:
Q = Q.sum(on = "countries", axis = 0)
qt = Q["all", ["all", sectors]]
qy = Q["all", ["all", hh_sectors]]

In [27]:
figaro.add_dimensions({
        "Emission categories" : crf_labs,
        "hh_sectors" : hh_sectors
    })
figaro.parts["qy"] = figaro.new_part(
    name="qy",
    dimensions = ["Emission categories",["countries", "hh_sectors"]]
)
figaro.parts["qt"] = figaro.new_part(
    name="qt",
    dimensions = ["Emission categories",["countries","sectors"]]
)

figaro_countries = [c for c in figaro.labels["countries"] if c != "ROW"]
figaro.qt[crf_labs, [figaro_countries, sectors]] = qt[crf_labs, [figaro_countries, sectors]]
figaro.qy[crf_labs, [figaro_countries, hh_sectors]] = qy[crf_labs, [figaro_countries, hh_sectors]]

save_to_nc(figaro, output_filepath)
print("Aggregation complete. Data saved to netCD.")

INFO:mrio_toolbox.mrio:Add dimension Emission categories to MRIO table
INFO:mrio_toolbox.mrio:Add dimension hh_sectors to MRIO table
INFO:mrio_toolbox.utils.savers._to_nc:Saving MRIO instance to /home/florian/Wissenschaft/eu-ets-inflation-impact/data/figaro io/formatted io/figaro_year2024_aggregated_eu27.nc
INFO:mrio_toolbox.utils.converters.xarray:Attribute metadata of part countries_grouped_t is of type <class 'dict'>, which is not compatible with xarray.
INFO:mrio_toolbox.utils.converters.xarray:Attribute groupings of part countries_grouped_t is of type <class 'dict'>, which is not compatible with xarray.
INFO:mrio_toolbox.utils.converters.xarray:Attribute parts of part countries_grouped_t is of type <class 'dict'>, which is not compatible with xarray.
INFO:mrio_toolbox.utils.converters.xarray:Attribute metadata of part countries_grouped_y is of type <class 'dict'>, which is not compatible with xarray.
INFO:mrio_toolbox.utils.converters.xarray:Attribute groupings of part countries_g

Aggregation complete. Data saved to netCD.
